In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [3]:
df = pd.read_parquet('../data/processed/merged.parquet')


In [4]:
df_products = df.drop_duplicates(subset='product_title').copy()


In [18]:
df_products['search_text'] = (
    df_products['product_title'].astype(str).fillna('') + ' ' +
    df_products['text'].astype(str).fillna('') + ' ' +
    df_products['description'].astype(str).fillna('')
).str.strip()

In [19]:
print(f"Products to index: {len(df_products)}")
print(df_products['search_text'].head(3))

Products to index: 110543
0    Helen of Troy 1511 Brush Iron, White, 3/4 Inch...
1    Miramist - Back Acne Spray for Body & Face - S...
2    GK HAIR Global Keratin Moisturizing Shampoo an...
Name: search_text, dtype: str


In [ ]:
# model = SentenceTransformer('all-MiniLM-L6-v2')
# embeddings = model.encode(
#     df_products['search_text'].tolist(),
#     show_progress_bar=True,
#     batch_size=64
# ).astype('float32')
# np.save("new_embeddings.npy", embeddings)
# print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1728 [00:00<?, ?it/s]

Embeddings shape: (110543, 384)


In [21]:
faiss.normalize_L2(embeddings)

In [22]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"Index size: {index.ntotal}")

Index size: 110543


In [24]:
faiss.write_index(index, '../data/processed/faiss_index/index_products.faiss')
df_products.to_parquet('../data/processed/new_products.parquet', index=False)
print("Saved new index and product dataframe.")

Saved new index and product dataframe.


In [44]:
df_products.head()

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details,combined_text,search_text
0,1.0,One Star,"Not what I wanted, wanted the 1 inch!",True,"Helen of Troy 1511 Brush Iron, White, 3/4 Inch...",3.9,16.3,"[Helen of Troy 1511 Brush Iron, White, 3/4 Inc...",Helen Of Troy,"{""Brand"": ""Helen Of Troy"", ""Hair Type"": ""All"",...","Helen of Troy 1511 Brush Iron, White, 3/4 Inch...","Helen of Troy 1511 Brush Iron, White, 3/4 Inch..."
1,5.0,Great product!,I've been using this on my son who has terribl...,True,Miramist - Back Acne Spray for Body & Face - S...,3.8,15.99,[Get back at acne with MiraMist topical clear ...,Infinite Health,"{""Item Form"": ""Liquid"", ""Skin Type"": ""All"", ""B...",Miramist - Back Acne Spray for Body & Face - S...,Miramist - Back Acne Spray for Body & Face - S...
2,5.0,Good Buy,My wife loves this item. She will continue to ...,True,GK HAIR Global Keratin Moisturizing Shampoo an...,4.8,None,[],GK HAIR,"{""Product Benefits"": ""Detangle, Color Protecti...",GK HAIR Global Keratin Moisturizing Shampoo an...,GK HAIR Global Keratin Moisturizing Shampoo an...
3,5.0,Results!,I’ve finally found lashes that look great and ...,True,Yiwamica Magnetic Eyelashes to Use with Magnet...,3.4,None,[],Yiwamica,"{""Brand"": ""Yiwamica"", ""Item Form"": ""Pair"", ""Mo...",Yiwamica Magnetic Eyelashes to Use with Magnet...,Yiwamica Magnetic Eyelashes to Use with Magnet...
4,3.0,It is ok but not great. One of the teeth broke...,I was hoping this would tame the frizz in my d...,True,EnPassion Hair Straightening Brush - For Gorge...,3.2,None,[],NaN,"{""Hair Type"": ""Frizzy"", ""Item Weight"": ""1.1 Po...",EnPassion Hair Straightening Brush - For Gorge...,EnPassion Hair Straightening Brush - For Gorge...


In [38]:
INDEX_PATH = "../data/processed/faiss_index/index_products.faiss"

_model = None
_index = None
_df = None

def load_semantic_artifacts():
    global _model, _index, _df

    if _model is None or _index is None or _df is None:
        _model = SentenceTransformer('all-MiniLM-L6-v2')
        _index = faiss.read_index(INDEX_PATH)
        _df = pd.read_parquet('../data/processed/new_products.parquet')

    return _model, _index, _df

def semantic_search(query, top_k=5):
    model, index, df = load_semantic_artifacts()

    query_embedding = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)  # normalize query too

    distances, indices = index.search(query_embedding, top_k)

    results = df.iloc[indices[0]].copy()
    results['score'] = distances[0]  # now it's cosine similarity (higher = better)

    final = results[["product_title", "details", "description","text", "score", "average_rating"]]
    final["description"] = final["description"].apply(
        lambda x: x[:200] + "..." if isinstance(x, str) and len(x) > 200 else x
    )

    return final

In [43]:
semantic_search("somthing for dry lips""")

,product_title,details,description,text,score,average_rating
102243,Sister Bees100% Natural Beeswax Lip Balm - Fes...,"{""Brand"": ""Sister Bees"", ""Item Form"": ""Balm"", ...",[],Beeswax is the first ingredient on the list. T...,0.742830,3.8
34655,6pcs Kleancolor Madly Matte Lipgloss Lip Gloss...,"{""Brand"": ""Kleancolor"", ""Skin Type"": ""All"", ""I...",[],Pretty colors but will make your lips very dry...,0.721391,3.3
28063,Quinn’s Organic Cotton Candy Sugar Lip Scrub E...,"{""Brand"": ""Quinns Cosmetics"", ""Item Form"": ""ba...",[],Works great,0.709035,4.3
40835,"Ruby Kisses Stix O' Miracle Lip Balm, Maximum ...","{""Brand"": ""Ruby Kisses"", ""Item Form"": ""Balm"", ...",[],Helps heal chapped/cracked lips! Highly recomm...,0.708717,4.3
67421,Beauticontrol Lip Balm 2 Piece Bundle,"{""UPC"": ""755332326648 794168350922"", ""Manufact...","[Softens, smoothes and helps protect lips from...",My favorite lip balm!,0.707611,4.7
